# 01 — Exploración de Datos (EDA)
**Proyecto:** Dashboard Predictivo del Mercado Laboral Colombiano
**Concurso:** Datos al Ecosistema 2026 — Reto 5: Economía y Empleo


## 1. Configuración e imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Rutas
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
print('Librerías cargadas correctamente.')

## 2. Carga de datos
### 2.1 Serie temporal DANE (tasas mensuales)

In [ ]:
df_serie = pd.read_csv(f'{PROCESSED_DIR}/serie_temporal_td.csv', parse_dates=['fecha'])
print(f'Forma: {df_serie.shape}')
print(f'Período: {df_serie["fecha"].min()} → {df_serie["fecha"].max()}')
df_serie.head(10)

### 2.2 Dataset SENA (datos.gov.co)

In [ ]:
df_sena = pd.read_csv(f'{RAW_DIR}/sena/sena_inscritos.csv')
print(f'Registros SENA: {len(df_sena)}')
print(f'Columnas: {list(df_sena.columns)}')
df_sena.head()

## 3. Estadísticas descriptivas
### 3.1 Serie temporal

In [ ]:
print('=== ESTADÍSTICAS SERIE TEMPORAL ===')
print(df_serie[['tasa_desocupacion','tasa_ocupacion','tasa_global_participacion']].describe().round(2))

### 3.2 Valores nulos y tipos de datos

In [ ]:
print('--- Nulos en serie temporal ---')
print(df_serie.isnull().sum())
print()
print('--- Tipos de datos ---')
print(df_serie.dtypes)

## 4. Visualización de la serie temporal

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

for ax, col, color, label in zip(
    axes,
    ['tasa_desocupacion', 'tasa_ocupacion', 'tasa_global_participacion'],
    ['#e74c3c', '#2ecc71', '#3498db'],
    ['Tasa de Desocupación (%)', 'Tasa de Ocupación (%)', 'TGP (%)']
):
    ax.plot(df_serie['fecha'], df_serie[col], color=color, linewidth=2, marker='o', markersize=4)
    ax.set_ylabel(label, fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
fig.suptitle('Indicadores del Mercado Laboral Colombiano (DANE GEIH)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada.')

## 5. Análisis de estacionalidad

In [ ]:
df_serie['mes'] = df_serie['fecha'].dt.month
df_serie['anio'] = df_serie['fecha'].dt.year

mensual = df_serie.groupby('mes')['tasa_desocupacion'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(mensual.index, mensual.values, color='#3498db', edgecolor='white', linewidth=0.8)
ax.set_xlabel('Mes')
ax.set_ylabel('TD promedio (%)')
ax.set_title('Estacionalidad promedio de la Tasa de Desocupación')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/estacionalidad.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top ocupaciones SENA

In [ ]:
col_ocupacion = [c for c in df_sena.columns if 'ocupaci' in c.lower()][0]
col_2019 = [c for c in df_sena.columns if '2019' in c][0]
col_2020 = [c for c in df_sena.columns if '2020' in c][0]

df_sena_clean = df_sena[[col_ocupacion, col_2019, col_2020]].copy()
df_sena_clean.columns = ['ocupacion', 'inscritos_2019', 'inscritos_2020']
df_sena_clean['total'] = df_sena_clean['inscritos_2019'].fillna(0) + df_sena_clean['inscritos_2020'].fillna(0)
top10 = df_sena_clean.nlargest(10, 'total')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top10['ocupacion'], top10['total'], color='#2ecc71', edgecolor='white')
ax.set_xlabel('Total inscritos (2019+2020)')
ax.set_title('Top 10 Ocupaciones por Inscritos SENA')
plt.tight_layout()
plt.savefig('../reports/figures/top_ocupaciones_sena.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total inscritos SENA: {df_sena_clean["total"].sum():,.0f}')

## 7. Conclusiones EDA

- La serie temporal tiene **34 observaciones** (mayo 2023 – febrero 2026), suficiente para Holt-Winters.
- Se observa una **tendencia bajista** sostenida: la TD pasó de ~11% a ~8.5%.
- La **estacionalidad anual** es clara: enero-febrero son los meses de mayor desempleo.
- El dataset SENA tiene **566 registros** de ocupaciones con inscritos 2019-2020.
- No hay valores nulos críticos en la serie temporal.